In [ ]:
# === ARRANQUE EN COLAB: arbol de carpetas de la sesion =====================
# Este cuaderno se escribio para correr desde la carpeta `notebook/` de su
# sesion, con ../data, ../figuras y ../resultados al lado. Colab arranca en
# /content y sin ese arbol, asi que aqui se recrea y nos situamos dentro: con
# eso, todas las rutas relativas del cuaderno funcionan igual que en local.
import os, sys

if "google.colab" in sys.modules:
    _RAIZ = "/content/E06_pronostico_causalidad"
    for _sub in ("notebook", "data", "figuras", "resultados"):
        os.makedirs(os.path.join(_RAIZ, _sub), exist_ok=True)
    os.chdir(os.path.join(_RAIZ, "notebook"))
    print("Colab: carpeta de trabajo en", os.getcwd())


# Sesión EPE E6 — Pronosticar el futuro y entender el porqué

**Curso "Herramientas de Ciencias de Datos" · Modalidad EPE · UPC · Facultad de Negocios**

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jonatanfigueroagil-creator/Herramientas-de-Ciencias-de-Datos/blob/master/Sesiones_EPE/E06_pronostico_causalidad/notebook/EPE_S6_series_causal.ipynb)

> Enfoque EPE: se prioriza la **intuición y la decisión de negocio** sobre el formalismo.
> Se trabajan el pronóstico de series temporales, la inferencia causal y el análisis de
> supervivencia con la **intuición**; se retiran los métodos técnicos pesados (identificación
> de órdenes ARIMA/SARIMA, DiD/PSM/variables instrumentales y la regresión de Cox formal), que
> quedan fuera del alcance de esta sesión.

## Objetivos de aprendizaje
Al terminar la sesión, el participante es capaz de:
1. **Pronosticar** una variable de negocio (ventas/demanda) identificando **tendencia, estacionalidad y ruido**, y leer un pronóstico con su **intervalo**.
2. **Validar** un pronóstico sin *fuga temporal* (sin usar información del futuro) y comparar dos métodos.
3. Distinguir **correlación de causalidad**: reconocer un **confusor** y por qué el **A/B** es el estándar de oro para decidir.
4. Leer una **curva de retención** (Kaplan-Meier) por segmento y convertirla en **CLV** para priorizar la retención de clientes.

## Mapa de la sesión
| # | Bloque | Datos |
|---|---|---|
| A | Pronosticar: tendencia, estacionalidad, ruido, intervalos y validación honesta | Store Item Demand (tienda 1 · producto 1) |
| B | Entender el porqué: correlación ≠ causalidad, confusor, A/B | Retención observacional |
| C | ¿Cuándo se van los clientes?: curva de retención (Kaplan-Meier) + CLV | Telco Customer Churn |

**Materiales hermanos:** guía de laboratorio `laboratorio/GUIA_LABORATORIO_E06.docx`,
plantillas `plantillas/plantilla_pronostico.docx` y `plantillas/guia_correlacion_causalidad.docx`,
ejercicios `evaluacion/drills.docx`, entregable `evaluacion/entregable.docx` y fuentes de actualidad
las fuentes de actualidad de la sesión.

In [ ]:
# SKIP-LOCAL: solo Colab.
# Colab ya trae el nucleo cientifico (numpy, pandas, scipy, matplotlib, seaborn,
# scikit-learn, statsmodels, openpyxl) COMPILADO ENTRE SI. Reinstalarlo con las
# versiones del venv del curso ROMPE el entorno: scipy y statsmodels dejan de
# importar con "cannot import name '_slice' from 'numpy._core.umath'". Por eso
# aqui solo se instala lo que Colab NO trae.
import sys

if "google.colab" in sys.modules:
    %pip install -q lifelines prophet

# Trazabilidad (sin reinstalar): versiones en uso frente a la matriz
# certificada del curso en la matriz de versiones certificada del curso. Si alguna difiere, las cifras
# pueden variar en los ultimos decimales; el metodo y las conclusiones no.
import importlib.metadata as _md

_CERTIFICADAS = {
    "matplotlib": "3.11.1",
    "numpy": "2.5.1",
    "openpyxl": "3.1.5",
    "pandas": "2.3.3",
    "statsmodels": "0.14.6",
}

print(f"{'paquete':18}{'en uso':14}{'certificada':14}estado")
for _p, _cert in _CERTIFICADAS.items():
    try:
        _v = _md.version(_p)
    except Exception:
        _v = "ausente"
    _estado = "=" if _v == _cert else "distinta (se respeta la de Colab)"
    print(f"{_p:18}{_v:14}{_cert:14}{_estado}")


In [ ]:
# Librerías de la sesión
import os, sys, io, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.exponential_smoothing.ets import ETSModel
from lifelines import KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test

# Estética (paleta de marca UPC: rojo + neutros)
UPC_RED, UPC_INK, UPC_GRAY, UPC_BLUE = "#E4002B", "#2D2D2D", "#9AA0A6", "#5B6770"
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "axes.titleweight": "bold",
                     "font.size": 11, "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
print("Librerías OK -", "Colab" if "google.colab" in sys.modules else "entorno local")

In [ ]:
# Rutas robustas (funcionan con nbconvert local y en Colab) y carga de datos.
import urllib.request, hashlib

def _base_dir():
    if "google.colab" in sys.modules:
        return os.getcwd()
    d = os.getcwd()
    for _ in range(6):
        if os.path.exists(os.path.join(d, "la ficha de la sesiónmd")):
            return d
        d = os.path.dirname(d)
    d = os.getcwd()
    return os.path.dirname(d) if os.path.basename(d).lower() == "notebook" else d

BASE = _base_dir()
DATA = os.getcwd() if "google.colab" in sys.modules else os.path.join(BASE, "data")
RES = os.path.join(BASE, "resultados"); FIG = os.path.join(BASE, "figuras")
os.makedirs(RES, exist_ok=True); os.makedirs(FIG, exist_ok=True); os.makedirs(DATA, exist_ok=True)
XLSX = os.path.join(RES, "E06_resultados.xlsx")

# Respaldo portable (Colab): si falta un archivo, se obtiene con cascada de fuentes.
# store1_item1: repo del autor -> mirror abierto original (filtrando tienda 1/producto 1)
# -> verificacion por nº de filas y checksum. telco: mirror abierto byte-identico del original.
REPO = ('https://raw.githubusercontent.com/jonatanfigueroagil-creator/'
        'Herramientas-de-Ciencias-de-Datos/master/Sesiones_EPE/E06_pronostico_causalidad/data/')
URL_TRAIN = ('https://raw.githubusercontent.com/jgonzalezab/'
             'Store-Item-Demand-Forecasting/master/Data/train.csv')
URL_TELCO = ('https://raw.githubusercontent.com/treselle-systems/'
             'customer_churn_analysis/master/WA_Fn-UseC_-Telco-Customer-Churn.csv')
STORE_ROWS = 1826           # tienda 1, producto 1: 2013-01-01..2017-12-31 (diario)
STORE_SHA256 = 'eac47a67b6c4e3204bb811791263ef8c174e628b46754f90a6c8d3c8af82a95e'

def _local(nombre):
    return os.path.join(DATA, nombre)

def _sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 16), b''):
            h.update(chunk)
    return h.hexdigest()

def cargar_store():
    ruta = _local("store1_item1.csv")
    if not os.path.exists(ruta):
        # Fuente 1: muestra ya filtrada en el repo del autor.
        try:
            print("Descargando muestra store1_item1.csv del repo del autor...")
            urllib.request.urlretrieve(REPO + "store1_item1.csv", ruta)
        except Exception as e:
            # Fuente 2: mirror abierto original; se filtra tienda 1 / producto 1.
            print("Repo del autor no disponible (%r); se usa el mirror abierto original." % e)
            full = pd.read_csv(URL_TRAIN)
            sub = full[(full["store"] == 1) & (full["item"] == 1)][["date", "store", "item", "sales"]]
            sub.reset_index(drop=True).to_csv(ruta, index=False, encoding="utf-8")
    df = pd.read_csv(ruta, parse_dates=["date"])
    # Verificacion de integridad (nº de filas siempre; checksum si la muestra es la canonica).
    assert len(df) == STORE_ROWS, "store1_item1.csv con nº de filas inesperado: %d" % len(df)
    csum = _sha256(ruta)
    print("store1_item1.csv OK (%d filas; SHA256 %s%s)" % (
        len(df), csum[:16], "= canonico" if csum == STORE_SHA256 else " != canonico (re-filtrado)"))
    return df

def cargar_telco():
    ruta = _local("telco_churn.csv")
    if not os.path.exists(ruta):
        print("Descargando telco_churn.csv del mirror abierto...")
        urllib.request.urlretrieve(URL_TELCO, ruta)
    return pd.read_csv(ruta)

def cargar_retencion():
    ruta = _local("retencion_observacional.csv")
    if not os.path.exists(ruta):
        print("Regenerando el escenario de retencion (DGP documentado)...")
        rng = np.random.default_rng(20260719)
        eng = rng.normal(0, 1, 5000); ten = rng.normal(24, 6, 5000)
        camp = (rng.uniform(0, 1, 5000) < 1/(1+np.exp(-(0.9*eng)))).astype(int)
        val = 50 + 8*camp + 20*eng + 3*ten + rng.normal(0, 8, 5000)
        pd.DataFrame({"engagement": eng, "tenure": ten, "campaign": camp,
                      "value": val}).to_csv(ruta, index=False, encoding="utf-8")
    return pd.read_csv(ruta)

store = cargar_store()
telco = cargar_telco()
reten = cargar_retencion()
print("Store Item Demand (tienda 1, producto 1):", store.shape)
print("Telco Customer Churn                    :", telco.shape)
print("Retencion observacional                 :", reten.shape)

---
## A. Pronosticar el futuro: tendencia, estacionalidad y ruido

Una **serie temporal** es una secuencia de valores ordenados en el tiempo. Se lee como la suma
de tres piezas: la **tendencia** (el nivel de fondo que sube o baja), la **estacionalidad** (un
patrón que se repite cada periodo fijo —p. ej. cada 12 meses—) y el **ruido** (lo que queda, sin
patrón). Separarlas evita confundir un **pico estacional** (diciembre) con **crecimiento real**.

Se trabaja la **demanda mensual** de un producto (tienda 1, producto 1) del reto *Store Item
Demand*. Se agregan las ventas diarias a **frecuencia mensual** y se **descompone** con STL.

> 💡 **Intuición de negocio.** Antes de pronosticar hay que *diagnosticar* de qué está hecha la
> serie: la **tendencia** dice *cuánto* crece el negocio y la **estacionalidad** dice *cuándo*
> prepararse. Confundir un pico de diciembre con crecimiento de fondo lleva a comprar inventario
> que no rota (o a incurrir en quiebre de stock en el pico).

**❓ Qué se quiere averiguar.** ¿De qué está hecha la demanda de este producto: crece de fondo, repite un patrón cada 12 meses, o ambos rasgos a la vez?

- **Qué decide:** la tendencia dice **cuánto** hay que comprar de más el año que viene; la estacionalidad dice **cuándo** reforzar inventario y personal. Confundir un pico de temporada con crecimiento real lleva a comprar mercancía que no rota.
- **Antes de mirar el resultado:** si el panel de **tendencia** sube y el de **estacionalidad** repite el mismo dibujo cada 12 meses, la serie tiene las dos piezas y el pronóstico necesitará ambas. Si la estacionalidad saliera plana, bastaría con planificar por nivel. Y si el panel de **ruido** mostrara estructura —una onda, un escalón—, la descomposición se habría quedado corta y quedaría señal sin explicar.

> 🔎 **Qué hace este código.** Se agrega la demanda diaria a **frecuencia mensual** (`resample("MS")`)
> y se separa con **STL** (`period=12`) en tres capas: observado = **tendencia + estacionalidad +
> ruido**. Al ejecutar, fijarse en los cuatro paneles de la figura, de arriba abajo.

In [ ]:
# Serie MENSUAL de demanda (agregando las ventas diarias) y descomposición en
# tendencia + estacionalidad + ruido (STL). Esta figura es EDA: se traza de los datos.
serie = store.set_index("date")["sales"].sort_index().resample("MS").sum()
serie.name = "ventas"
stl = STL(serie, period=12, robust=True).fit()

print(f"Serie mensual: {serie.index.min():%Y-%m} a {serie.index.max():%Y-%m}  ({len(serie)} meses)")
print(f"Tendencia: pasa de {stl.trend.iloc[0]:.0f} a {stl.trend.iloc[-1]:.0f} unidades/mes (crece).")
pico = int(stl.seasonal.groupby(serie.index.month).mean().idxmax())
print(f"Estacionalidad: amplitud ~{stl.seasonal.max()-stl.seasonal.min():.0f} unidades; pico en el mes {pico} (mitad de año).")

fig, ax = plt.subplots(4, 1, figsize=(9, 7.5), sharex=True)
ax[0].plot(serie.index, serie.values, color=UPC_INK); ax[0].set_ylabel("Observado")
ax[0].set_title("Demanda mensual = tendencia + estacionalidad + ruido")
ax[1].plot(stl.trend.index, stl.trend.values, color=UPC_RED); ax[1].set_ylabel("Tendencia")
ax[2].plot(stl.seasonal.index, stl.seasonal.values, color=UPC_BLUE); ax[2].set_ylabel("Estacional")
ax[3].plot(stl.resid.index, stl.resid.values, color=UPC_GRAY); ax[3].set_ylabel("Ruido")
ax[3].axhline(0, color=UPC_INK, lw=0.6)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "E06_descomposicion.png"), bbox_inches="tight")
plt.show()

📖 **Cómo se lee esta salida.** La demanda **crece** de fondo (tendencia ascendente) y además tiene un
**patrón anual** claro: sube a mitad de año y baja en los extremos. El panel de **ruido** oscila
alrededor de cero sin estructura evidente. Para planear inventario y personal, la tendencia dice
*cuánto crece el negocio* y la estacionalidad dice *cuándo* prepararse; confundirlas lleva a
acumular exceso de inventario en temporada baja o a incurrir en quiebre de stock en el pico.

> 💡 **Intuición.** Cuando la amplitud del patrón anual **crece con el nivel** de la serie, la
> estacionalidad es *multiplicativa* (conviene el logaritmo); si es constante, es *aditiva*. Aquí
> el ruido es pequeño frente a los otros dos componentes: la serie es **muy predecible**.

**❓ Qué se quiere averiguar.** ¿Algún método supera a la regla de menor costo que existe —repetir el mismo mes del año pasado—, y con qué margen?

- **Qué decide:** si la respuesta es no, el pronóstico se hace con la línea base y no se paga licencia ni mantenimiento por un modelo. Ninguna herramienta se compra por su prestigio: se compra por la ventaja que obtiene frente al *naive estacional*.
- **Antes de mirar el resultado:** el sMAPE se lee **menor es mejor**. Si un método quedara **por encima** del naive, resulta prescindible. Si quedara **muy por debajo**, se justifica lo que cuesta. Y si la ventaja fuese de **décimas de punto sobre un único holdout de 12 meses**, lo honesto es declarar **empate dentro del ruido** en lugar de declarar un modelo superior. Conviene además anticipar el resultado de Prophet, el más sofisticado de los tres: esta serie está agregada a mensual y no tiene tramos de tendencia ni festivos móviles, así que compite fuera de su terreno.

> 🔎 **Qué hace este código.** La validación **separa en el tiempo**: entrena con el pasado (`train`)
> y mide el error solo en los **últimos 12 meses** (`test`), que el modelo nunca vio. Compara tres
> métodos por **sMAPE** (menor es mejor) contra la línea base *naive estacional*.

> 💡 **De dónde proviene Prophet — qué preguntaba el paper.** Taylor y Letham publicaron *Forecasting at
> Scale* (2018) para resolver un cuello de botella de **organización**, no para mejorar la precisión:
> la demanda de pronósticos fiables superaba con mucho el ritmo al que un número reducido de especialistas
> podía producirlos, y quien conocía el negocio no tenía formación en series temporales. Por eso
> eligieron como criterio la **reparabilidad** antes que la exactitud —un pronóstico malo se tolera
> si alguien puede corregirlo, y los órdenes de un ARIMA no se traducen al lenguaje del negocio— y
> plantearon el modelo como un **ajuste de curva** con tres mandos que un gerente sabe contestar:
> cuándo cambió el negocio, qué ciclos tiene y qué días son especiales. Los propios autores acotan su
> propuesta a series con tendencia por tramos, estacionalidad múltiple y festivos móviles; esta serie
> está **agregada a mensual** y no presenta ninguno de los tres rasgos, así que Prophet compite fuera
> de su terreno. Que quede detrás **confirma** su criterio: la herramienta se elige por las
> características de la serie, no por el prestigio del método.

In [ ]:
# Validar SIN hacer trampa: se separa en el TIEMPO (entrenar con el pasado, probar con
# los ultimos 12 meses). Nunca se mezcla el futuro con el pasado (nada de barajar).
h = 12
train, test = serie.iloc[:-h], serie.iloc[-h:]

def smape(a, f):
    a, f = np.asarray(a, float), np.asarray(f, float)
    return float(np.mean(2*np.abs(a-f)/(np.abs(a)+np.abs(f)))*100)
def mape(a, f):
    a, f = np.asarray(a, float), np.asarray(f, float)
    return float(np.mean(np.abs(a-f)/np.abs(a))*100)

# (1) Linea base obligatoria: naive-estacional = repetir el mismo mes del año anterior.
pred_snaive = pd.Series(train.iloc[-12:].values, index=test.index)

# (2) Suavizamiento exponencial (Holt-Winters): da mas peso a lo reciente + estacionalidad.
ets_tr = ETSModel(train.astype(float), error='add', trend='add',
                  seasonal='add', seasonal_periods=12).fit(disp=False)
pr = ets_tr.get_prediction(start=len(train), end=len(train)+h-1).summary_frame(alpha=0.05)
pred_ets = pd.Series(pr['mean'].values, index=test.index)
low_ets = pd.Series(pr['pi_lower'].values, index=test.index)
up_ets  = pd.Series(pr['pi_upper'].values, index=test.index)

# (3) Prophet: herramienta moderna que separa tendencia + estacionalidad automaticamente.
prophet_smape = np.nan
try:
    from prophet import Prophet
    dfp = train.reset_index(); dfp.columns = ['ds', 'y']
    mp = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
    mp.fit(dfp)
    fc = mp.predict(mp.make_future_dataframe(periods=h, freq='MS')).set_index('ds').iloc[-h:]
    pred_prophet = pd.Series(fc['yhat'].values, index=test.index)
    prophet_smape = smape(test.values, pred_prophet.values)
    prophet_mape = mape(test.values, pred_prophet.values)
except Exception as e:
    print('Prophet no disponible en este entorno:', repr(e))
    prophet_mape = np.nan

cobertura = float(np.mean((test.values >= low_ets.values) & (test.values <= up_ets.values))*100)
# Se reportan DOS metricas de error a proposito: sMAPE y MAPE. Comparar ambas evita
# coronar un "campeon" que solo gana con una metrica.
metricas = pd.DataFrame({
    'modelo': ['Naive estacional (línea base)', 'Suavizamiento exponencial (Holt-Winters)', 'Prophet'],
    'sMAPE_%': [smape(test.values, pred_snaive.values), smape(test.values, pred_ets.values), prophet_smape],
    'MAPE_%':  [mape(test.values, pred_snaive.values),  mape(test.values, pred_ets.values),  prophet_mape],
})
print(metricas.to_string(index=False))
print(f'\nCobertura del intervalo 95% (Holt-Winters) en el holdout: {cobertura:.0f}% (ideal ~95%).')

# Lectura HONESTA del "campeon": el ranking DEPENDE de la metrica y por eso no es concluyente.
gana_smape = metricas.iloc[:2].sort_values('sMAPE_%')['modelo'].iloc[0]
gana_mape  = metricas.iloc[:2].sort_values('MAPE_%')['modelo'].iloc[0]
brecha = abs(metricas.loc[1, 'sMAPE_%'] - metricas.loc[0, 'sMAPE_%'])
print('\n--- Que modelo "gana"? Depende de la metrica ---')
print(f"  Por sMAPE gana: {gana_smape.split('(')[0].strip()} ({metricas.loc[1,'sMAPE_%']:.3f} vs {metricas.loc[0,'sMAPE_%']:.3f}).")
print(f"  Por MAPE  gana: {gana_mape.split('(')[0].strip()} ({metricas.loc[0,'MAPE_%']:.3f} vs {metricas.loc[1,'MAPE_%']:.3f}).")
print(f"  El ranking se INVIERTE segun la metrica y la brecha en sMAPE es de solo {brecha:.2f} pp")
print("  sobre un UNICO holdout de 12 meses: Holt-Winters y el naive son INDISTINGUIBLES")
print("  dentro del ruido. Lo robusto es que Prophet (8.31) si queda claramente detras.")

📖 **Cómo se lee esta salida.** La regla de oro es **validar con datos retenidos**: se entrena con el
pasado y se mide el error sobre los **últimos 12 meses**, que el modelo no vio. Barajar los datos
(como en otros problemas) produciría *fuga temporal*: usaría el futuro para predecir el pasado y daría
una precisión ilusoria que **cae de forma abrupta en producción**.

Todo modelo se compara contra una **línea base** simple (el *naive estacional*: repetir el mismo
mes del año anterior). Aquí conviene mirar **dos métricas** y ser honestos: el ranking **depende de
la métrica**. Por **sMAPE**, el suavizamiento exponencial (Holt-Winters) queda ligeramente por
delante (**4.92 % vs 5.10 %**); pero por **MAPE el naive queda por delante** (**5.003 % vs 5.076 %**). La brecha
es menor a **0.2 puntos** sobre un **único holdout de 12 meses**: Holt-Winters y el naive son
**indistinguibles dentro del ruido**, y declarar superior a uno de ellos sería sobreinterpretar los datos
(decidirlo con firmeza pediría *backtesting de origen móvil*, que queda fuera del alcance de esta
sesión). Lo que **sí** es robusto —y es la conclusión de negocio— es que **Prophet**, más sofisticado,
queda **claramente detrás** (8.31 %): **una herramienta más compleja no es automáticamente mejor**.

> ⚠️ **Alerta nº 1: fuga temporal.** En una serie **nunca** se barajan los datos (nada de
> partición aleatoria ni validación cruzada estándar): mezclar meses deja que el modelo "vea el
> futuro", el error reportado resulta **irrealmente bajo** y luego el desempeño cae de forma abrupta en producción. La
> validación honesta respeta el orden del tiempo: entrenar con el pasado, medir en el futuro reservado.
>
> ⚠️ **Alerta nº 2: olvidar la línea base y la estacionariedad.** Un modelo solo "vale" si
> **supera al naive estacional** (y aquí, con una sola métrica y un solo holdout, esa ventaja no es
> concluyente). Una serie con tendencia/estación **no es estacionaria** (su nivel cambia en el
> tiempo); por eso se valida con un tramo futuro real y no se extrapola sin validación.

**❓ Qué se quiere averiguar.** ¿Con cuánta incertidumbre se puede prometer la demanda del mes 12, comparada con la del mes 1?

- **Qué decide:** el compromiso de inventario. Un pronóstico puntual sin intervalo obliga a planificar como si el número fuera cierto; con el intervalo se elige con qué límite abastecer según qué error cueste más, el faltante o el sobrante.
- **Antes de mirar el resultado:** si el ancho (`upper − lower`) saliera **igual** en el mes 1 y en el mes 12, el modelo afirmaría que predecir dentro de un año es tan fiable como predecir el mes que viene, y habría que desconfiar de él. Lo esperable es que **crezca con el horizonte**: cuanto más lejos se mira, más ancho el rango y más caro comprometerse.

> 🔎 **Qué hace este código.** Se reentrena el suavizamiento con **toda** la serie y se proyectan
> **12 meses hacia adelante** con su **intervalo del 95 %**. Observar cómo el ancho del intervalo
> (`upper − lower`) **crece** del mes 1 al mes 12: más horizonte, más incertidumbre.

In [ ]:
# Pronóstico a 12 meses HACIA ADELANTE con su intervalo del 95% (Holt-Winters,
# reentrenado con toda la serie). El intervalo comunica la incertidumbre accionable.
ets_full = ETSModel(serie.astype(float), error='add', trend='add',
                    seasonal='add', seasonal_periods=12).fit(disp=False)
ff = ets_full.get_prediction(start=len(serie), end=len(serie)+11).summary_frame(alpha=0.05)
fut_idx = pd.date_range(serie.index[-1] + pd.offsets.MonthBegin(), periods=12, freq='MS')
futuro = pd.DataFrame({'fecha': fut_idx, 'pred': ff['mean'].values,
                       'lower': ff['pi_lower'].values, 'upper': ff['pi_upper'].values})
anchos = (futuro['upper'] - futuro['lower']).round(0)
print('Pronóstico del primer mes:  %.0f  [%.0f , %.0f]' % (
      futuro['pred'].iloc[0], futuro['lower'].iloc[0], futuro['upper'].iloc[0]))
print('Pronóstico del mes 12   :  %.0f  [%.0f , %.0f]' % (
      futuro['pred'].iloc[-1], futuro['lower'].iloc[-1], futuro['upper'].iloc[-1]))
print('Ancho del intervalo: crece de %.0f (mes 1) a %.0f (mes 12) unidades.' % (anchos.iloc[0], anchos.iloc[-1]))

📖 **Cómo se lee esta salida.** El pronóstico nunca es un solo número: viene con un **intervalo** que
dice *qué grado de certeza hay*. Ese intervalo **se ensancha con el horizonte** —predecir el mes
que viene es más preciso que dentro de un año—. Para decidir inventario se usa el **límite
superior** del intervalo (para no incurrir en quiebre de stock en el pico) o el **inferior** (para no
acumular exceso de inventario), según qué error cueste más. Un pronóstico puntual sin intervalo está incompleto.

> ⚠️ **Alerta: reportar solo "el número".** Entregar el pronóstico puntual sin su intervalo
> oculta la incertidumbre y provoca decisiones de inventario frágiles. Como el intervalo **se
> ensancha** con el horizonte, cuanto más lejos se mira menos certeza hay: se planifica con el
> **rango**, no con la línea central.

**❓ Qué se quiere averiguar.** El intervalo dice cuánta incertidumbre hay, pero no cuántas unidades pedir. ¿Cuántas se piden cuando **faltar** cuesta más que **sobrar**?

- **Qué decide:** la orden de compra del mes 1, en unidades. Es el punto donde el pronóstico deja de ser un gráfico y se convierte en dinero comprometido.
- **Antes de mirar el resultado:** el **ratio crítico** `CR = Cu/(Cu+Co)` fija el percentil de la demanda al que conviene abastecer. Con `Cu = 5` y `Co = 2` sale `CR ≈ 0,71`, por encima del 50 %: `Q*` debe quedar **por encima** del pronóstico puntual. Si los dos costes fueran iguales, `CR = 0,5` y se pediría exactamente el punto; si sobrar fuese lo caro, `CR < 0,5` y se pediría **por debajo**. El pronóstico no cambia en ninguno de los tres casos: lo que mueve la cantidad es el precio relativo de los dos errores.

> 🔎 **Qué hace este código (opcional, negocio).** El intervalo dice *cuánta incertidumbre* hay, pero
> no *cuánto pedir*. El **modelo del vendedor de periódicos** (*newsvendor*) traduce el intervalo en
> una **cantidad óptima**: cuando **faltar** cuesta más que **sobrar**, conviene abastecer **por
> encima** del pronóstico puntual. La cantidad se fija en el **percentil crítico** `CR = Cu/(Cu+Co)`
> de la demanda (`Cu` = costo de faltante, `Co` = costo de sobrante). Es la regla que hace operativo
> el «usar el límite superior o el inferior según qué error cueste más». *(Los costos son
> ilustrativos; lo que se enseña es el método.)*

In [ ]:
# Del intervalo a la CANTIDAD a pedir: el modelo del "vendedor de periodicos" (newsvendor).
# El intervalo no dice cuanto pedir; eso lo fija el COSTO RELATIVO de los dos errores:
#   - faltante (underage) Cu: margen que se pierde por cada unidad que falto (venta perdida);
#   - sobrante  (overage)  Co: costo por cada unidad que sobro (mantener/rebajar/mermar).
# Regla optima: abastecer al PERCENTIL critico  CR = Cu / (Cu + Co)  de la demanda.
# Los costos de abajo son ILUSTRATIVOS (dependen del negocio); lo que se enseña es el metodo.
from scipy.stats import norm

Cu, Co = 5.0, 2.0                                   # USD/unidad (ilustrativos): faltar cuesta mas que sobrar
CR = Cu / (Cu + Co)                                 # ratio critico -> percentil de abastecimiento
z = float(norm.ppf(CR))
mu1 = float(futuro['pred'].iloc[0])                 # pronostico puntual del mes 1
sigma1 = float((futuro['upper'].iloc[0] - futuro['lower'].iloc[0]) / (2*1.959964))  # sigma implicita del IC 95%
Q_star = mu1 + z * sigma1                           # cantidad optima a pedir (mes 1)

newsvendor = pd.DataFrame({
    'concepto': ['Costo de faltante Cu (USD/unid, ilustrativo)',
                 'Costo de sobrante Co (USD/unid, ilustrativo)',
                 'Ratio critico Cu/(Cu+Co)',
                 'Percentil de abastecimiento z (normal)',
                 'Pronostico puntual mes 1 (unid)',
                 'Desv. implicita del IC 95% (unid)',
                 'Cantidad optima Q* mes 1 (unid)'],
    'valor': [Cu, Co, round(CR, 3), round(z, 3), round(mu1, 1), round(sigma1, 1), round(Q_star, 1)],
})
print(newsvendor.to_string(index=False))
print(f'\nComo faltar cuesta mas que sobrar (Cu>Co), el ratio critico es {CR:.0%}: se abastece POR ENCIMA')
print(f'del pronostico puntual ({mu1:.0f}) -> Q* = {Q_star:.0f} unidades (percentil {CR:.0%} del intervalo).')
print('Si sobrar fuera lo caro (Co>Cu) el ratio bajaria de 50% y se pediria por DEBAJO del punto.')

---
## B. Entender el porqué: correlación ≠ causalidad

Que dos variables **suban juntas** (correlación) no significa que una **cause** la otra. Una empresa
de suscripción lanzó una **campaña de retención** y quiere saber cuánto valor **añadió**. El
problema: la campaña **no** se asignó al azar —marketing contactó a los clientes ya más activos
(*engagement* alto)—, y esos clientes **de por sí** valen más. El *engagement* es un **confusor**:
empuja a la vez la campaña y el valor, e **infla** la comparación ingenua.

Este escenario de negocio tiene la ventaja didáctica de que su **efecto verdadero es conocido**
(+8 USD por diseño): permite ver *cuánto se equivoca* la comparación ingenua.

> ⚠️ **Alerta: "subió después" no es "subió porque".** La correlación puede venir de un
> **confusor**, una tercera variable que mueve a la vez la causa y el efecto. Caso típico: la
> campaña se lanza en temporada alta, así que la **estacionalidad** —no la campaña— empuja las
> ventas. *(Los métodos formales para aislar el efecto —DiD, matching, variables instrumentales—
> quedan fuera del alcance EPE; aquí basta la intuición del confusor y del A/B.)*

> 🔎 **Qué hace este código.** Primero la comparación **ingenua** (diferencia de medias campaña vs.
> no campaña); luego se comprueba que `engagement` correlaciona con **ambos** (el confusor) y se
> **ajusta** comparando clientes similares. El resultado se contrasta con el efecto **verdadero** (+8 USD).

In [ ]:
# (1) Comparación INGENUA: valor medio de quienes recibieron la campaña vs. quienes no.
naive = reten.loc[reten.campaign==1, 'value'].mean() - reten.loc[reten.campaign==0, 'value'].mean()

# El confusor: engagement se relaciona CON la campaña y CON el valor.
corr_camp = reten['engagement'].corr(reten['campaign'])
corr_val  = reten['engagement'].corr(reten['value'])

# (2) Comparar CLIENTES SIMILARES: se 'controla' el confusor (regresión, tema de E2).
mod_aj = smf.ols('value ~ campaign + engagement + tenure', data=reten).fit()
ajustado = mod_aj.params['campaign']
verdadero = 8.0  # efecto real por diseño del escenario (solo se conoce en un experimento/simulación)

# (3) INCERTIDUMBRE del efecto ajustado: error estándar e intervalo de confianza al 95%.
# El punto no basta: hay que saber si +7.85 es distinguible del ruido (y de la verdad +8.0).
ajustado_se = mod_aj.bse['campaign']
ic = mod_aj.conf_int(alpha=0.05).loc['campaign']
ajustado_ic_low, ajustado_ic_up = float(ic[0]), float(ic[1])
ajustado_t, ajustado_p = mod_aj.tvalues['campaign'], mod_aj.pvalues['campaign']

print(f'Efecto INGENUO de la campaña (diferencia de medias): +{naive:.2f} USD')
print(f'  -> engagement correlaciona con recibir campaña: {corr_camp:.2f}  y con el valor: {corr_val:.2f}  (confusor)')
print(f'Efecto AJUSTADO (comparando clientes similares)    : +{ajustado:.2f} USD')
print(f'  -> SE = {ajustado_se:.2f} ; IC95% = [{ajustado_ic_low:.2f}, {ajustado_ic_up:.2f}] ; t = {ajustado_t:.1f} ; p = {ajustado_p:.1e}')
print(f'Efecto VERDADERO (conocido por diseño)             : +{verdadero:.2f} USD')
print(f'\nLa comparación ingenua exagera el efecto ~{naive/verdadero:.1f} veces.')
print(f'El IC95% [{ajustado_ic_low:.2f}, {ajustado_ic_up:.2f}] CONTIENE a +8.00: el efecto ajustado es')
print('estadisticamente INDISTINGUIBLE del efecto verdadero (estimacion creible, no prueba).')

📖 **Cómo se lee esta salida.** La comparación ingenua dice que la campaña añade **+23.6 USD**, pero su
efecto verdadero es **+8 USD**: la exagera **~3 veces**. La causa es el **confusor** *engagement*,
que infla la diferencia porque los clientes contactados ya valían más. Al **comparar clientes
similares** (controlando el confusor) el efecto baja a **+7.85 USD**, cerca de la verdad. Y como el
punto no basta, se reporta su **incertidumbre**: **SE ≈ 0.25** e **IC 95 % = [7.36, 8.34]**. Ese
intervalo **contiene +8.00**, de modo que el efecto ajustado es **indistinguible del verdadero**:
es una estimación *creíble*, no una prueba.

Por eso el **A/B test es el estándar de oro**: al asignar la campaña **al azar**, los grupos
quedan iguales en todo lo demás (incluido el *engagement*), y la diferencia observada **sí** mide
la causa —sin necesidad de ajustar nada—. Es el mismo principio del A/B de *Cookie Cats* visto en
E1. Regla práctica para decidir presupuesto: **cuando se pueda experimentar, se experimenta**;
cuando no, se buscan y controlan los confusores, con humildad sobre lo que no se midió.

> 💡 **Intuición del confusor.** El efecto ingenuo (+23.6) triplica al verdadero (+8) porque
> **mezcla** el efecto de la campaña con el hecho de que se dirigió a clientes que ya valían más.
> "Comparar similares" —o repartir al azar en un A/B— desactiva ese sesgo.

> ⚠️ **Alerta al comparar en el tiempo (DiD).** Un atajo frecuente es mirar antes-vs-después con
> un grupo de control (diferencias en diferencias). Solo es creíble si ambos grupos venían con
> **tendencias paralelas** antes de la acción; si ya divergían, el "efecto" estimado es un artefacto.

---
## C. ¿Cuándo se van los clientes? Curva de retención (Kaplan-Meier) y CLV

Predecir *si* un cliente se irá responde a medias; el negocio necesita saber **cuándo** para
intervenir a tiempo. El **análisis de supervivencia** modela el **tiempo-hasta-evento** (aquí el
*churn*) tratando bien a los **clientes aún activos**: no se sabe cuándo se irán, solo que
**siguen** al cortar los datos (dato *censurado*). La **curva de Kaplan-Meier** `S(t)` es la
**curva de retención**: la fracción de clientes que sigue activa pasado el mes `t`.

Se usa **Telco Customer Churn**: `tenure` = meses hasta el evento, `Churn` = si se dio de baja.
Se comparan segmentos por **tipo de contrato**.

> ⚠️ **Alerta: descartar a los clientes activos.** Los que **siguen** al cortar los datos son
> *censurados*, no "no-eventos": descartarlos o tratarlos como bajas sesga la curva. Kaplan-Meier usa
> su información parcial. El supuesto que lo sostiene es la **censura no informativa**: que un
> cliente siga activo no debe estar ligado a su riesgo de irse (si el corte se debe a la propia baja
> inminente, el método induce a error).

> 🔎 **Qué hace este código.** Se codifica `evento` = churn y la duración = `tenure`; se estima la
> **curva de retención** `S(t)` por tipo de contrato con `KaplanMeierFitter`, se contrasta con la
> prueba de **log-rank** y se convierte cada curva en **CLV** descontando el margen mes a mes.

> 💡 **De dónde proviene la curva — qué preguntaba el paper.** Kaplan y Meier (1958) no buscaban una
> curva escalonada: atacaban la **censura**, el dato de quien sale de observación sin que se vea su
> evento. Llegaron al problema por separado y desde mundos distintos —uno medía la vida de los tubos
> de vacío de los repetidores de cables telefónicos submarinos, el otro venía de un artículo sobre la
> duración del cáncer—, prueba de que la censura no es un asunto clínico sino de **cualquier** dato
> de tiempo hasta un evento, la antigüedad de un cliente activo incluida. Eligieron un **producto de
> probabilidades condicionales** porque con censura el número de sujetos en riesgo cambia a cada
> instante y una proporción simple deja de ser calculable, y evitaron suponer una forma para `S(t)`
> porque no había base para creer que un cable submarino y un paciente compartieran una familia de
> curvas. Catorce años después, **Cox (1972)** respondió la pregunta siguiente —el efecto de varias
> variables sobre el tiempo hasta el evento— y modeló el **riesgo instantáneo** en lugar del tiempo,
> justamente porque el riesgo se define «dado que el sujeto sigue vivo», que es lo que aporta un
> censurado. Ese modelo aquí solo se nombra.

In [ ]:
# Preparación: evento = churn (1/0), duración = tenure (meses). Los 'activos' son
# datos CENSURADOS (siguen sin evento). Se descartan 11 clientes con tenure = 0.
tel = telco.copy()
tel['TotalCharges'] = pd.to_numeric(tel['TotalCharges'], errors='coerce')
tel = tel[tel['tenure'] > 0].copy()
tel['evento'] = (tel['Churn'] == 'Yes').astype(int)
print(f"Clientes: {len(tel)}  |  churn (eventos): {tel['evento'].sum()} ({tel['evento'].mean()*100:.1f}%)  |  activos (censurados): {(1-tel['evento']).sum()}")

segmentos = ['Month-to-month', 'One year', 'Two year']
km = KaplanMeierFitter()
curvas = {}       # S(t) mes a mes por segmento (para exportar y graficar)
filas_clv = []
meses = list(range(0, 73))
d = 0.01          # descuento mensual (1%)
for seg in segmentos:
    sub = tel[tel['Contract'] == seg]
    km.fit(sub['tenure'], sub['evento'])
    curvas[seg] = [float(km.predict(t)) for t in meses]
    m = float(sub['MonthlyCharges'].mean())
    clv = sum(m * float(km.predict(t)) / (1 + d)**t for t in range(1, 73))
    filas_clv.append({'segmento': seg, 'margen_mes': round(m, 2),
                      'S_12m': round(float(km.predict(12)), 3),
                      'S_24m': round(float(km.predict(24)), 3),
                      'S_72m': round(float(km.predict(72)), 3),
                      'CLV_72m': round(clv, 2), 'n': int(len(sub))})
clv_tab = pd.DataFrame(filas_clv)
km_tab = pd.DataFrame({'mes': meses, **{seg: curvas[seg] for seg in segmentos}})

lr = multivariate_logrank_test(tel['tenure'], tel['Contract'], tel['evento'])
print(f'\nPrueba de log-rank (¿difieren las curvas?): chi2 = {lr.test_statistic:.0f}, p = {lr.p_value:.1e}')
print(clv_tab.to_string(index=False))

📖 **Cómo se lee esta salida.** Las curvas de retención de los tres contratos son **muy distintas** (la
prueba de log-rank lo confirma: p ≈ 0). El contrato **mes-a-mes** retiene solo **~70 % al año** y
cae a **~13 % a 6 años**; el de **dos años** mantiene **~94 %** a 6 años. La sorpresa está en el
**CLV**: mes-a-mes tiene el **margen mensual más alto** (66 USD) pero el **CLV más bajo** (~1 804
USD), porque su baja retención lo anula. **La retención domina el valor.**

Decisión: **priorizar la migración** de los clientes mes-a-mes hacia contratos anuales; el techo
de gasto de esa oferta lo fija el **diferencial de CLV** (casi +1 300 USD por cliente migrado).

> ⚠️ **Atención: el CLV no es monótono en la retención.** El de **un año** tiene el CLV **más alto**
> (~3 105 USD), por encima del de **dos años** (~3 097 USD), **pese a que el bianual retiene mejor**
> (S(72) ≈ 94 % vs ≈ 57 %). No es contradicción: el CLV mezcla **retención × margen × descuento**.
> El contrato anual cobra un **margen mensual algo mayor** (65.1 vs 60.9 USD) y, con **descuento del
> 1 % mensual sobre 72 meses**, los meses lejanos —donde vive la ventaja de retención del bianual—
> pesan poco (1.01⁷² ≈ 2.05, el mes 72 vale la mitad). Resultado: ambos CLV quedan casi **iguales**
> (~0.3 % de diferencia). La lección de negocio se mantiene —migrar del mes-a-mes rinde ~+1 300 USD—,
> pero entre anual y bianual el CLV **no** ordena por retención sola.

> 💡 **Intuición: la retención domina el valor.** El cliente mes-a-mes paga más *por mes* pero se
> va pronto, así que su CLV es el más bajo. El valor de un cliente no es lo que paga hoy, sino
> **cuánto tiempo se queda**: retener y alargar contratos protege más ingreso que maximizar el margen.

> ⚠️ **Alerta al leer riesgos (mención).** Si más adelante se modela el riesgo con una regresión
> de Cox, su **hazard ratio (HR) no es un riesgo relativo** ni una probabilidad: es cuánto
> multiplica la tasa **instantánea** de irse. Además el HR se supone **constante en el tiempo**
> (hazards proporcionales); si las curvas de dos grupos **se cruzan**, ese supuesto falla.
> *(La regresión de Cox formal queda fuera del alcance EPE; aquí basta la curva de retención y el CLV.)*

In [ ]:
# --- Exportar TODOS los resultados a resultados/E06_resultados.xlsx ---
holdout = pd.DataFrame({'fecha': test.index, 'real': test.values,
    'ets_pred': pred_ets.values, 'ets_lower': low_ets.values, 'ets_upper': up_ets.values,
    'naive_estacional': pred_snaive.values})
with pd.ExcelWriter(XLSX, engine='openpyxl') as w:
    holdout.to_excel(w, sheet_name='pronostico_holdout', index=False)
    futuro.to_excel(w, sheet_name='pronostico_futuro', index=False)
    metricas.round(3).to_excel(w, sheet_name='forecast_metricas', index=False)
    pd.DataFrame({'estimador': ['ingenuo (naive)', 'ajustado (clientes similares)', 'verdadero (diseño)'],
                  'valor_usd': [round(naive, 2), round(ajustado, 2), verdadero]}
                 ).to_excel(w, sheet_name='causalidad', index=False)
    km_tab.round(4).to_excel(w, sheet_name='km_contrato', index=False)
    clv_tab.to_excel(w, sheet_name='clv_contrato', index=False)
    # Hojas NUEVAS (no alteran las de negocio anteriores):
    # incertidumbre del efecto causal ajustado + traza del newsvendor.
    pd.DataFrame({'metrica': ['efecto_ajustado_usd', 'error_estandar', 'ic95_inferior',
                              'ic95_superior', 't', 'p_value'],
                  'valor': [round(ajustado, 2), round(ajustado_se, 2), round(ajustado_ic_low, 2),
                            round(ajustado_ic_up, 2), round(ajustado_t, 2), ajustado_p]}
                 ).to_excel(w, sheet_name='causalidad_ic', index=False)
    newsvendor.to_excel(w, sheet_name='newsvendor', index=False)
print('Excel de resultados escrito en:', XLSX)
print('Hojas:', ['pronostico_holdout','pronostico_futuro','forecast_metricas',
                 'causalidad','km_contrato','clv_contrato','causalidad_ic','newsvendor'])

In [ ]:
# --- Figuras de RESULTADOS: se generan LEYENDO el Excel (convención del curso) ---
ho = pd.read_excel(XLSX, sheet_name='pronostico_holdout')
fu = pd.read_excel(XLSX, sheet_name='pronostico_futuro')
mt = pd.read_excel(XLSX, sheet_name='forecast_metricas')
ca = pd.read_excel(XLSX, sheet_name='causalidad')
kmx = pd.read_excel(XLSX, sheet_name='km_contrato')
cl = pd.read_excel(XLSX, sheet_name='clv_contrato')

# R1) Pronóstico + intervalo: holdout (real vs Holt-Winters) y futuro con banda.
fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.plot(ho['fecha'], ho['real'], 'o-', color=UPC_INK, label='Real (holdout)')
ax.plot(ho['fecha'], ho['ets_pred'], 's--', color=UPC_RED, label='Pronóstico (Holt-Winters)')
ax.fill_between(ho['fecha'], ho['ets_lower'], ho['ets_upper'], color=UPC_RED, alpha=0.15, label='Intervalo 95%')
ax.plot(fu['fecha'], fu['pred'], 's--', color=UPC_RED)
ax.fill_between(fu['fecha'], fu['lower'], fu['upper'], color=UPC_RED, alpha=0.15)
ax.axvline(ho['fecha'].iloc[0], color=UPC_GRAY, ls=':', lw=1)
ax.set_title('Pronóstico de demanda con intervalo del 95% (el intervalo se ensancha)')
ax.set_ylabel('Ventas/mes'); ax.legend(loc='upper left', fontsize=9)
fig.tight_layout(); fig.savefig(os.path.join(FIG, 'E06_pronostico_intervalo.png'), bbox_inches='tight'); plt.show()

# R2) Comparación de métodos por sMAPE (validación honesta).
fig, ax = plt.subplots(figsize=(7, 3.8))
col = [UPC_GRAY, UPC_RED, UPC_BLUE]
b = ax.barh(mt['modelo'], mt['sMAPE_%'], color=col)
ax.bar_label(b, fmt='%.2f%%', padding=3); ax.invert_yaxis()
ax.set_title('Error en datos retenidos (sMAPE, menor es mejor)'); ax.set_xlabel('sMAPE (%)')
fig.tight_layout(); fig.savefig(os.path.join(FIG, 'E06_forecast_metricas.png'), bbox_inches='tight'); plt.show()

# R3) Causalidad: ingenuo vs ajustado vs verdadero.
fig, ax = plt.subplots(figsize=(6.4, 3.8))
col3 = [UPC_GRAY, UPC_RED, UPC_INK]
b = ax.bar(ca['estimador'], ca['valor_usd'], color=col3)
ax.bar_label(b, fmt='+%.1f', padding=3)
ax.set_title('El efecto de la campaña: la comparación ingenua lo triplica')
ax.set_ylabel('Efecto (USD)'); ax.set_xticklabels(ca['estimador'], fontsize=8.5)
fig.tight_layout(); fig.savefig(os.path.join(FIG, 'E06_causalidad.png'), bbox_inches='tight'); plt.show()

# R4) Curvas de retención (Kaplan-Meier) por contrato.
fig, ax = plt.subplots(figsize=(7.6, 4.2))
colores = {'Month-to-month': UPC_RED, 'One year': UPC_BLUE, 'Two year': UPC_INK}
for seg in ['Month-to-month', 'One year', 'Two year']:
    ax.step(kmx['mes'], kmx[seg]*100, where='post', color=colores[seg], label=seg)
ax.axhline(50, color=UPC_GRAY, ls=':', lw=1)
ax.set_title('Curva de retención por tipo de contrato (Kaplan-Meier)')
ax.set_xlabel('Meses (tenure)'); ax.set_ylabel('Retención S(t) (%)'); ax.legend()
fig.tight_layout(); fig.savefig(os.path.join(FIG, 'E06_km_contrato.png'), bbox_inches='tight'); plt.show()

# R5) CLV por segmento (la retención domina el valor).
fig, ax = plt.subplots(figsize=(6.6, 3.8))
b = ax.bar(cl['segmento'], cl['CLV_72m'], color=[colores[s] for s in cl['segmento']])
ax.bar_label(b, fmt='%.0f', padding=3)
ax.set_title('CLV a 72 meses por contrato (mes-a-mes: más margen, menos valor)')
ax.set_ylabel('CLV (USD)')
fig.tight_layout(); fig.savefig(os.path.join(FIG, 'E06_clv_contrato.png'), bbox_inches='tight'); plt.show()
print('Figuras de resultados guardadas en:', FIG)

---
### Ejercicios (drills)
Los enunciados completos están en `evaluacion/drills.docx` (se resuelven en parejas; entrega individual):
1. **Pronóstico a 12 meses:** leer el intervalo del pronóstico (¿qué mes es más incierto y por qué?).
2. **Validar y comparar dos métodos** con datos retenidos: ¿cuál supera a la línea base y cómo se justifica?
3. **Curva de retención:** comparar dos segmentos de clientes y traducirlo a una decisión de retención.

El **entregable evaluable** (`evaluacion/entregable.docx`, rúbrica vigesimal = 20) integra los tres
bloques: un pronóstico con intervalo + validación, una lectura de correlación vs. causalidad, y una
curva de retención con su CLV. Las plantillas `plantillas/plantilla_pronostico.docx` y
`plantillas/guia_correlacion_causalidad.docx` materializan el trabajo.

### Cierre y para seguir explorando

**Tres ideas para la gerencia:** (1) un pronóstico sin **intervalo** está incompleto y la validación
debe ser **honesta** (probar con el futuro, superar una línea base simple); (2) **correlación no es
causa** —cuidado con el confusor; el **A/B** es el estándar de oro para decidir inversión—; (3) en
retención, importa **cuándo** se van los clientes: la **curva de retención** y el **CLV** priorizan
dónde intervenir. Estas piezas alimentan el **proyecto integrador** (del problema al pronóstico y a
la recomendación accionable).

**Fuentes de actualidad** (detalle y enlaces verificados en las fuentes de actualidad de la sesión):
- *On the retraining frequency of global models in retail demand forecasting* — arXiv/MLwA (2025-2026): cada cuánto reentrenar un modelo de demanda sin perder precisión.
- *Causal Marketing: The Future of Data-Driven Decision-Making* — Measured (08/07/2025): «qué pasó DESPUÉS» no es «qué pasó PORQUE».
- *Average ecommerce subscription churn by billing period: the 2026 benchmark* — Eightx (29/05/2026): curvas de retención y CLV por periodo de facturación.